# 拉脱法测液体表面张力系数计算器

本 Notebook 用于通过力敏传感器测定拉脱法实验中液体（水）的表面张力系数 $\alpha$。包含砝码电压标定（线性回归求传感器灵敏度 $K$）、拉脱峰值电压与吊环内外径统计、不确定度合成及规范修约。

---
### 实验原理与数学公式

#### 1. 表面张力系数计算公式
当薄壁圆环从液体表面被拉脱的一瞬间，所受的最大液体表面张力与圆环内外周长成正比：
$$\alpha = \frac{F}{\pi (d_1 + d_2)}$$
* $F$：最大拉脱力（通过力敏传感器测得 $F = U_{\mathrm{pull}} / K$）
* $K$：力敏传感器灵敏度 $(\mathrm{V/N})$ 或 $(\mathrm{mV/N})$
* $d_1, d_2$：金属圆环的内径与外径

#### 2. 力敏传感器灵敏度标定
使用单重为 $m$ 的标准小砝码增重与减重 $i$ 个（$i = 0, 1, \dots, 5$），各测得对应的输出电压 $U_{1,i}, U_{2,i}$。取增减重平均值：
$$\bar{U}_i = \frac{U_{1,i} + U_{2,i}}{2}, \quad F_i = i \cdot m \cdot g$$
使用最小二乘线性回归对 $(F_i, \bar{U}_i)$ 拟合直线 $U = K \cdot F + b$，斜率即为灵敏度 $K$，其标准不确定度记为 $u(K)$。

#### 3. 不确定度合成与传递
* 拉脱力 $F$ 的相对不确定度：
$$u_r(F) = \sqrt{ \left(\frac{u_U}{\bar{U}_{\mathrm{pull}}}\right)^2 + \left(\frac{u_K}{K}\right)^2 }$$
* 直径和 $(d_1 + d_2)$ 的不确定度：
$$u(d_1 + d_2) = \sqrt{u^2(d_1) + u^2(d_2)}, \quad u_r(d_1+d_2) = \frac{u(d_1 + d_2)}{d_1 + d_2}$$
* 表面张力系数 $\alpha$ 的总相对不确定度与绝对不确定度：
$$u_r(\alpha) = \sqrt{u_r^2(F) + u_r^2(d_1 + d_2)}, \quad u(\alpha) = \alpha \cdot u_r(\alpha)$$

In [ ]:
import math
from decimal import Decimal
from python.utils import scientific_round, calculate_stats, linear_regression

print("表面张力系数模块加载完成。")

### 1. 传感器砝码标定数据输入与拟合
> **提示**：可在此输入单个小砝码质量 `m_mass_g` 以及增重、减重各 6 个点对应的标定电压。

In [ ]:
G = Decimal("9.80")          # 重力加速度 (m/s^2)
PI = Decimal(str(math.pi))

# 单个小砝码质量 (g)
m_mass_g = Decimal("0.50")
m_mass_kg = m_mass_g / Decimal("1000")

# 标定电压 (0~5个砝码, 单位与后续拉脱电压保持统一即可, 如 V 或 mV)
U_calib_1 = [Decimal("0.00"), Decimal("0.052"), Decimal("0.104"), Decimal("0.155"), Decimal("0.207"), Decimal("0.259")]  # 增重
U_calib_2 = [Decimal("0.00"), Decimal("0.051"), Decimal("0.103"), Decimal("0.156"), Decimal("0.208"), Decimal("0.260")]  # 减重

# 计算平均电压与外力
U_calib_avg = [(U_calib_1[i] + U_calib_2[i]) / Decimal("2") for i in range(6)]
F_calib = [Decimal(str(i)) * m_mass_kg * G for i in range(6)]

# 线性回归拟合灵敏度 K
K, K_b, r, u_K = linear_regression(F_calib, U_calib_avg)

print("=" * 45)
print("           传 感 器 标 定 拟 合 结 果         ")
print("=" * 45)
print(f"灵敏度斜率 K   : {K:.6f} (电压单位/N)")
print(f"截距 b         : {K_b:.6f}")
print(f"相关系数 r     : {r:.6f}")
print(f"斜率不确定度 u_K: {u_K:.6e}")

### 2. 拉脱实验数据与圆环几何尺寸输入

In [ ]:
# 仪器允差
delta_U = Decimal("0.001")   # 电压表仪器误差
delta_d = Decimal("0.02")    # 游标卡尺仪器误差 (mm)

# 6次测量数据列表
U_pull_vals = [Decimal("0.215"), Decimal("0.214"), Decimal("0.216"), Decimal("0.215"), Decimal("0.213"), Decimal("0.215")]
d1_vals = [Decimal("33.10"), Decimal("33.12"), Decimal("33.08"), Decimal("33.10"), Decimal("33.14"), Decimal("33.10")]  # 内径 (mm)
d2_vals = [Decimal("35.20"), Decimal("35.22"), Decimal("35.18"), Decimal("35.20"), Decimal("35.22"), Decimal("35.20")]  # 外径 (mm)

# 统计各量
U_pull_mean, U_pull_u = calculate_stats(U_pull_vals, delta_U)[:2]
d1_mean, d1_u = calculate_stats(d1_vals, delta_d)[:2]
d2_mean, d2_u = calculate_stats(d2_vals, delta_d)[:2]

print(f"拉脱平均电压 U_pull: {U_pull_mean:.4f} ± {U_pull_u:.4f}")
print(f"圆环平均内径 d1    : {d1_mean:.3f} ± {d1_u:.3f} mm")
print(f"圆环平均外径 d2    : {d2_mean:.3f} ± {d2_u:.3f} mm")

### 3. 表面张力系数计算与不确定度修约

In [ ]:
# 计算拉脱力 F (N)
F_pull_mean = U_pull_mean / K

# 换算为标准国际单位 m
d1_m = d1_mean / Decimal("1000")
d2_m = d2_mean / Decimal("1000")
d1_u_m = d1_u / Decimal("1000")
d2_u_m = d2_u / Decimal("1000")
d_sum = d1_m + d2_m

# 表面张力系数 α = F / [π * (d1 + d2)]
alpha_val = F_pull_mean / (PI * d_sum)

# 不确定度合成传递
u_r_F_sq = (U_pull_u / U_pull_mean)**2 + (u_K / K)**2
u_r_F = Decimal(str(math.sqrt(float(u_r_F_sq))))

u_d_sum = Decimal(str(math.sqrt(float(d1_u_m**2 + d2_u_m**2))))
u_r_d_sum = u_d_sum / d_sum

u_r_alpha = Decimal(str(math.sqrt(float(u_r_F**2 + u_r_d_sum**2))))
alpha_u = alpha_val * u_r_alpha

# 科学修约
alpha_final, u_final = scientific_round(alpha_val, alpha_u)

print("=" * 45)
print("         表 面 张 力 系 数 计 算 结 果        ")
print("=" * 45)
print(f"最大拉脱力 F     : {F_pull_mean:.6f} N")
print(f"圆环周长项 (d1+d2): {(d_sum*1000):.3f} mm")
print(f"表面张力系数 α (原始): {alpha_val:.6f} N/m")
print("-" * 45)
print(f"相对不确定度 u_r : {u_r_alpha * 100:.2f}%")
print(f"绝对不确定度 u_α : {alpha_u:.6f} N/m")
print("-" * 45)
print(f"★ 最终修约结果   : α = {alpha_final} ± {u_final} N/m")
print("=" * 45)